# WaterVision AI: YOLO26n-cls turbidity classification
This is a workflow scaffold, not the original research notebook. Verify data rights and family-aware splits before use. Outputs are not potability or field-safety assessments.

In [ ]:
%pip install -q ultralytics scikit-learn pandas matplotlib


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
PROJECT_DIR = Path('/content/drive/MyDrive/WaterVisionAI')
DATA_DIR = Path('/content/watervision_data')  # set to authorized train/<class>, val/<class> folders
print('Project folder:', PROJECT_DIR, 'Dataset ready:', DATA_DIR.exists())

## Confirm prepared splits
This validates directory structure only; it does not validate family-level separation. Use the original family manifest for CV.

In [ ]:
for split in ('train', 'val'):
    path = DATA_DIR / split
    assert path.is_dir(), f'Missing {path}'
    print(split, [p.name for p in path.iterdir() if p.is_dir()])

## Train a demonstration model
This generic run is not a reproduction of the reported CV. Train each family-aware fold separately to reproduce that experiment, retaining distinct checkpoints.

In [ ]:
from ultralytics import YOLO
model = YOLO('yolo26n-cls.pt')
model.train(data=str(DATA_DIR), epochs=50, imgsz=224, batch=32, device=0, name='watervision_demo')

## Inference with a specific checkpoint
Choose a checkpoint and record its fold identity. Do not combine or relabel fold checkpoints as a unified model.

In [ ]:
from pathlib import Path
from ultralytics import YOLO
WEIGHTS = Path('/content/runs/classify/watervision_demo/weights/best.pt')
IMAGE = Path('/content/example.jpg')  # change to an authorized image
assert WEIGHTS.is_file() and IMAGE.is_file()
result = YOLO(str(WEIGHTS)).predict(source=str(IMAGE), imgsz=224)[0]
for i in result.probs.data.argsort(descending=True).tolist():
    print(result.names[i], float(result.probs.data[i]))